*Practicing a supervised algorithm **Random Forest Classifier** in a **functional style** to predict yes/no*

In [37]:
#dependencies
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import(
    GridSearchCV,
    cross_val_score,
    train_test_split
)

from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [38]:
#declaring the constants
N_ESTIMATORS=100
RANDOM_STATE=42
CV_FOLDS=5
TEST_SIZE=0.2

PARAM_GRID={
    "model__n_estimators":[100, 200],
    "model__max_depth":[10, 20, None],
    "model__max_features":["sqrt","log2"],
    "model__min_samples_split":[2, 5],
}

In [39]:
#loding data
def load_data():
   data=load_breast_cancer()
   X=data.data
   y=data.target
   print(f"shape:{X.shape},{y.shape}")
   print(f"columns:{list(data.feature_names)}")
   print(f"target {list(data.target_names)}")
   print(f"hit rate:{y.mean() *100:.1f}")
   return X, y, list(data.feature_names), list(data.target_names)

In [40]:
#training data
def train_data(X, y):
   X_train, X_test, y_train, y_test=train_test_split(
      X, y, test_size=TEST_SIZE,
      random_state=RANDOM_STATE,
      stratify=y
   )
   print(f"train size:{y_train.shape[0]}")
   print(f"test size:{y_test.shape[0]}")
   return X_train, X_test, y_train, y_test

In [41]:
#wrapping the model,scaler inside the pipeline
def the_pipeline():
   return Pipeline([
       ("scaler", StandardScaler()),
       ("model", RandomForestClassifier(
           n_estimators=N_ESTIMATORS,
           random_state=RANDOM_STATE,
           n_jobs=-1,
       )),
   ])

In [42]:
#the cross validation point
def cross_validation(pipeline, X, y):
   scores=cross_val_score(
       pipeline, X, y,
       scoring="accuracy",
       cv=CV_FOLDS,
   )
   print(f"mean/average:{scores.mean():.2f}")
   print(f"standard deviation/consistency:{scores.std():.2f}")
   return scores

In [43]:
#tuning with GridSearchCV to choose the best of model
def tune_model(pipeline, X_train, y_train):
    grid_search = GridSearchCV(
        estimator  = pipeline,
        param_grid = PARAM_GRID,
        cv         = CV_FOLDS,
        scoring    = "accuracy",
        n_jobs     = -1,
        verbose    = 1,
    )
    grid_search.fit(X_train, y_train)
    print(f"best parameter combo:{grid_search.best_params_}")
    print(f"best score:{grid_search.best_score_:.2f}")
    return grid_search

In [46]:
#evaluating our model's performance
def evaluate_model(model, X_test, y_test, class_names):
    y_pred=model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"accuracy:{accuracy:.2f}")
    print(f"classification report:\n{classification_report(y_test, y_pred, target_names=class_names)}")
    return y_pred

In [47]:
def main():
    X, y, feature_names, class_names = load_data()
    X_train, X_test, y_train, y_test = train_data(X, y)

    pipeline=the_pipeline()
    cross_validation(pipeline, X_train, y_train)
    model=tune_model(pipeline, X_train, y_train)
    best_model=model.best_estimator_

    evaluate_model(best_model, X_test, y_test, class_names)

if __name__ == "__main__":
    main()

shape:(569, 30),(569,)
columns:[np.str_('mean radius'), np.str_('mean texture'), np.str_('mean perimeter'), np.str_('mean area'), np.str_('mean smoothness'), np.str_('mean compactness'), np.str_('mean concavity'), np.str_('mean concave points'), np.str_('mean symmetry'), np.str_('mean fractal dimension'), np.str_('radius error'), np.str_('texture error'), np.str_('perimeter error'), np.str_('area error'), np.str_('smoothness error'), np.str_('compactness error'), np.str_('concavity error'), np.str_('concave points error'), np.str_('symmetry error'), np.str_('fractal dimension error'), np.str_('worst radius'), np.str_('worst texture'), np.str_('worst perimeter'), np.str_('worst area'), np.str_('worst smoothness'), np.str_('worst compactness'), np.str_('worst concavity'), np.str_('worst concave points'), np.str_('worst symmetry'), np.str_('worst fractal dimension')]
target [np.str_('malignant'), np.str_('benign')]
hit rate:62.7
train size:455
test size:114
mean/average:0.95
standard devi